In [ ]:

!pip install kaggle
!pip install xgboost
!kaggle datasets download -d ellipticco/elliptic-data-set
!unzip elliptic-data-set.zip

In [1]:
import os
import requests
import zipfile
import pandas as pd
import torch
import torch.nn.functional as F
from torch_geometric.data import Data
from torch_geometric.nn import GCNConv
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report
from tqdm import tqdm

/opt/anaconda3/envs/gnn/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:

# ============================================================
# 2. Load Dataset
# ============================================================

features = pd.read_csv(
    "elliptic_bitcoin_dataset/elliptic_txs_features.csv",
    header=None
)

edges = pd.read_csv(
    "elliptic_bitcoin_dataset/elliptic_txs_edgelist.csv"
)

classes = pd.read_csv(
    "elliptic_bitcoin_dataset/elliptic_txs_classes.csv"
)

# Rename columns
features.rename(columns={0: "txId", 1: "time_step"}, inplace=True)
classes.rename(columns={"class": "label"}, inplace=True)

# Merge labels
data_df = features.merge(classes, on="txId")

# Filter labeled data (remove "unknown")
data_df = data_df[data_df.label != "unknown"]

# Convert labels
data_df["label"] = data_df["label"].map({"1": 1, "2": 0})

# Map txId to index
txid_to_idx = {txid: idx for idx, txid in enumerate(data_df.txId)}

# Filter edges to only include labeled nodes
edges = edges[
    edges.txId1.isin(txid_to_idx.keys()) &
    edges.txId2.isin(txid_to_idx.keys())
]

edge_index = torch.tensor([
    [txid_to_idx[row.txId1] for _, row in edges.iterrows()],
    [txid_to_idx[row.txId2] for _, row in edges.iterrows()]
], dtype=torch.long)

# Features
x = torch.tensor(
    data_df.iloc[:, 2:-1].values,
    dtype=torch.float
)

# Labels
y = torch.tensor(
    data_df.label.values,
    dtype=torch.long
)


In [3]:

# # Train/Test split
# train_idx, test_idx = train_test_split(
#     torch.arange(len(y)),
#     test_size=0.2,
#     stratify=y,
#     random_state=42
# )

# train_mask = torch.zeros(len(y), dtype=torch.bool)
# test_mask = torch.zeros(len(y), dtype=torch.bool)
# train_mask[train_idx] = True
# test_mask[test_idx] = True

# data = Data(x=x, edge_index=edge_index, y=y)
# data.train_mask = train_mask
# data.test_mask = test_mask

# print(data)


In [4]:
# Extract time steps
time_steps = torch.tensor(data_df.time_step.values)

# Example temporal split:
# Train on first 34 time steps
# Test on later time steps

train_mask = time_steps <= 34
test_mask = time_steps > 34

data = Data(x=x, edge_index=edge_index, y=y)
data.train_mask = train_mask
data.test_mask = test_mask

In [5]:

# ============================================================
# 3. Define GCN Model
# ============================================================

class GCN(torch.nn.Module):
    def __init__(self, input_dim, hidden_dim, num_classes):
        super(GCN, self).__init__()
        self.conv1 = GCNConv(input_dim, hidden_dim)
        self.conv2 = GCNConv(hidden_dim, num_classes)

    def forward(self, x, edge_index):
        x = self.conv1(x, edge_index)
        x = F.relu(x)
        x = self.conv2(x, edge_index)
        return x

model_GCN = GCN(
    input_dim=data.num_node_features,
    hidden_dim=64,
    num_classes=2
)

In [6]:
from torch_geometric.nn import SAGEConv
import torch.nn.functional as F
import torch

class GraphSAGE(torch.nn.Module):
    def __init__(self, input_dim, hidden_dim, num_classes):
        super(GraphSAGE, self).__init__()
        self.conv1 = SAGEConv(input_dim, hidden_dim)
        self.conv2 = SAGEConv(hidden_dim, num_classes)

    def forward(self, x, edge_index):
        x = self.conv1(x, edge_index)
        x = F.relu(x)
        x = F.dropout(x, p=0.3, training=self.training)
        x = self.conv2(x, edge_index)
        return x

model_GraphSAGE = GraphSAGE(
    input_dim=data.num_node_features,
    hidden_dim=64,
    num_classes=2
)


In [7]:
from torch_geometric.nn import GATConv

class GAT(torch.nn.Module):
    def __init__(self, input_dim, hidden_dim, num_classes):
        super(GAT, self).__init__()
        self.conv1 = GATConv(input_dim, hidden_dim, heads=4, dropout=0.3)
        self.conv2 = GATConv(hidden_dim * 4, num_classes, heads=1)

    def forward(self, x, edge_index):
        x = self.conv1(x, edge_index)
        x = F.elu(x)
        x = self.conv2(x, edge_index)
        return x

model_GAT = GAT(
    input_dim=data.num_node_features,
    hidden_dim=32,
    num_classes=2
)


In [8]:
def train(model):
    model.train()
    optimizer.zero_grad()
    out = model(data.x, data.edge_index)
    loss = criterion(out[data.train_mask], data.y[data.train_mask])
    loss.backward()
    optimizer.step()
    return loss.item()

def test(model):
    model.eval()
    out = model(data.x, data.edge_index)
    pred = out.argmax(dim=1)

    correct = (pred[data.test_mask] == data.y[data.test_mask]).sum()
    acc = int(correct) / int(data.test_mask.sum())

    return acc, pred


In [ ]:
from sklearn.metrics import classification_report, accuracy_score
import torch

# ============================================================
# Models
# ============================================================

models = {
    "GCN": lambda: GCN(data.num_node_features, 64, 2),
    "GAT": lambda: GAT(data.num_node_features, 32, 2),
    "GraphSAGE": lambda: GraphSAGE(data.num_node_features, 64, 2)
}

results = {}

# ============================================================
# Training Loop
# ============================================================

for name, model_fn in models.items():

    print(f"\n========== Training {name} ==========\n")

    # Fresh model instance
    model = model_fn()

    # Optimizer
    optimizer = torch.optim.Adam(model.parameters(), lr=0.003)

    # -------- Weighted Loss (Balanced) --------
    class_counts = torch.bincount(data.y[data.train_mask])
    total = class_counts.sum().float()
    weights = total / (2.0 * class_counts.float())

    criterion = torch.nn.CrossEntropyLoss(weight=weights)

    # -------- Training --------
    for epoch in range(1, 101):
        model.train()
        optimizer.zero_grad()

        out = model(data.x, data.edge_index)
        loss = criterion(out[data.train_mask], data.y[data.train_mask])

        loss.backward()
        optimizer.step()

    # -------- Evaluation --------
    model.eval()
    with torch.no_grad():
        out = model(data.x, data.edge_index)

        probs = torch.softmax(out, dim=1)[:, 1]
        threshold = 0.6
        pred = (probs > threshold).long()

    # Use SAME pred for everything
    y_true = data.y[data.test_mask].cpu().numpy()
    y_pred = pred[data.test_mask].cpu().numpy()

    report = classification_report(y_true, y_pred, output_dict=True)

    fraud_recall = report['1']['recall']
    fraud_f1 = report['1']['f1-score']
    acc = accuracy_score(y_true, y_pred)

    # Save for ranking
    results[name] = {
        "accuracy": acc,
        "fraud_recall": fraud_recall,
        "fraud_f1": fraud_f1
    }

    # Print report (using SAME y_pred)
    print(classification_report(y_true, y_pred))
    print(f"Model {name} completed.")

    print("=" * 60)

# ============================================================
# Ranking
# ============================================================

print("\n🏆 Ranking by Fraud F1-score:")
for k, v in sorted(results.items(), key=lambda x: x[1]["fraud_f1"], reverse=True):
    print(f"{k}: F1={v['fraud_f1']:.4f}, Recall={v['fraud_recall']:.4f}, Acc={v['accuracy']:.4f}")

print("\n🚨 Ranking by Fraud Recall:")
for k, v in sorted(results.items(), key=lambda x: x[1]["fraud_recall"], reverse=True):
    print(f"{k}: Recall={v['fraud_recall']:.4f}, F1={v['fraud_f1']:.4f}, Acc={v['accuracy']:.4f}")



========== Training GCN ==========

              precision    recall  f1-score   support

           0       0.97      0.92      0.95     15587
           1       0.37      0.63      0.46      1083

    accuracy                           0.90     16670
   macro avg       0.67      0.78      0.71     16670
weighted avg       0.93      0.90      0.92     16670

Model GCN completed.

========== Training GAT ==========

              precision    recall  f1-score   support

           0       0.98      0.84      0.91     15587
           1       0.24      0.74      0.37      1083

    accuracy                           0.84     16670
   macro avg       0.61      0.79      0.64     16670
weighted avg       0.93      0.84      0.87     16670

Model GAT completed.

========== Training GraphSAGE ==========

              precision    recall  f1-score   support

           0       0.98      0.92      0.95     15587
           1       0.37      0.71      0.49      1083

    accuracy           

In [11]:
import xgboost as xgb
from sklearn.metrics import classification_report, accuracy_score
import numpy as np

# Convert tensors to numpy
X = data.x.cpu().numpy()
y = data.y.cpu().numpy()

# Train/Test split using masks
X_train = X[data.train_mask.cpu().numpy()]
y_train = y[data.train_mask.cpu().numpy()]

X_test = X[data.test_mask.cpu().numpy()]
y_test = y[data.test_mask.cpu().numpy()]

# Handle imbalance automatically
scale_pos_weight = (y_train == 0).sum() / (y_train == 1).sum()

# Initialize XGBoost
model_xgb = xgb.XGBClassifier(
    n_estimators=200,
    max_depth=6,
    learning_rate=0.05,
    subsample=0.8,
    colsample_bytree=0.8,
    scale_pos_weight=scale_pos_weight,
    eval_metric="logloss",
    use_label_encoder=False,
    random_state=42
)

# Train
model_xgb.fit(X_train, y_train)

# Predict
y_pred = model_xgb.predict(X_test)

# Metrics
acc = accuracy_score(y_test, y_pred)
report = classification_report(y_test, y_pred, output_dict=True)

fraud_recall = report['1']['recall']
fraud_f1 = report['1']['f1-score']

print("\n========== XGBoost Results ==========\n")
print(f"Accuracy: {acc:.4f}")
print(f"Fraud Recall: {fraud_recall:.4f}")
print(f"Fraud F1-score: {fraud_f1:.4f}")
print("\nFull Classification Report:")
print(classification_report(y_test, y_pred))


/opt/anaconda3/envs/gnn/lib/python3.10/site-packages/xgboost/training.py:200: UserWarning: [23:17:39] WARNING: /Users/runner/work/xgboost/xgboost/src/learner.cc:782: 
Parameters: { "use_label_encoder" } are not used.

  bst.update(dtrain, iteration=i, fobj=obj)



========== XGBoost Results ==========

Accuracy: 0.9673
Fraud Recall: 0.7396
Fraud F1-score: 0.7462

Full Classification Report:
              precision    recall  f1-score   support

           0       0.98      0.98      0.98     15587
           1       0.75      0.74      0.75      1083

    accuracy                           0.97     16670
   macro avg       0.87      0.86      0.86     16670
weighted avg       0.97      0.97      0.97     16670



In [12]:
results["XGBoost"] = {
    "accuracy": acc,
    "fraud_recall": fraud_recall,
    "fraud_f1": fraud_f1
}


# =============================================
# Ranking by Fraud F1
# =============================================

print("\n🏆 Ranking by Fraud F1-score:")
for k, v in sorted(results.items(), key=lambda x: x[1]["fraud_f1"], reverse=True):
    print(f"{k}: F1={v['fraud_f1']:.4f}, Recall={v['fraud_recall']:.4f}")

# =============================================
# Ranking by Fraud Recall
# =============================================

print("\n🚨 Ranking by Fraud Recall:")
for k, v in sorted(results.items(), key=lambda x: x[1]["fraud_recall"], reverse=True):
    print(f"{k}: Recall={v['fraud_recall']:.4f}, F1={v['fraud_f1']:.4f}")



🏆 Ranking by Fraud F1-score:
XGBoost: F1=0.7462, Recall=0.7396
GraphSAGE: F1=0.4898, Recall=0.7128
GCN: F1=0.4624, Recall=0.6297
GAT: F1=0.3676, Recall=0.7378

🚨 Ranking by Fraud Recall:
XGBoost: Recall=0.7396, F1=0.7462
GAT: Recall=0.7378, F1=0.3676
GraphSAGE: Recall=0.7128, F1=0.4898
GCN: Recall=0.6297, F1=0.4624
